In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 固定局部视频时间 warp：已批准的一次六臂诊断
自动下载固定版本源码，无需上传ZIP。六臂、幅度1 latent格点、after44、150 Transformer/6 FP32 VAE、3600秒；L4 24GB或更高BF16设备。无自动重试或额外解锁开关，已有输出拒绝覆盖。


In [ ]:
from pathlib import Path
import sys, subprocess
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_COMMIT = 'f54917a30233c4ec03c75514f012ff5d86c7b94b'
SOURCE = Path('/content/local_temporal_warp_source')
if SOURCE.exists():
    raise FileExistsError('Use a fresh runtime; preserve the existing source directory')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
actual = subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip()
if actual != SOURCE_COMMIT:
    raise RuntimeError('Unexpected source revision')
print('Source:', actual)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','diffusers==0.35.2','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True)
subprocess.run(['ffmpeg','-version'],check=True)
# Keep Colab CUDA PyTorch; no model is loaded in this cell.


## 一次 GPU 入口
保持固定prompt、seed及配置；不要调整幅度、mask、step或observer救结果。超时/失败保留全部身份；最终主体、背景、形变与独立参照需要人工审阅，不能仅因q有效宣称运动。


In [ ]:
OUTPUT=Path('/content/drive/MyDrive/Video-WM/stage2-local-temporal-warp/run01')
for target in (OUTPUT,OUTPUT.parent/(OUTPUT.name+'.execution.log'),OUTPUT.parent/(OUTPUT.name+'.execution_exit.json')):
    if target.exists(): raise FileExistsError(str(target))
command=[sys.executable,'-m','experiments.stage2.run_local_temporal_warp','--config',str(SOURCE/'configs/stage2_local_temporal_warp.json'),'--output',str(OUTPUT)]
result=subprocess.run(command,cwd=SOURCE)
print('launcher exit',result.returncode)
result.check_returncode()


## 独立 CPU 外评
每臂每stream会生成31行 `reference_annotation_template.json`，p默认null。两遍分别复制文件、独立填写REVIEWED/实际轮廓中心/半径及shape/background/quality判读，不能用命令或q填点。只读CLI输出保持原31行/30邻接；参照未确认时不计算实际位移。例：
`python -m experiments.stage2.evaluate_local_temporal_warp --stream RUN/X_PLUS/mp4 --annotations independently_reviewed.json --output NEW_EVAL_DIR --off-stream RUN/OFF1/mp4`
全部实际结果须保留；CPU方案不授权扩样或后续GPU。
